# 17 — Semi-Autoregressive Drafting

**Engineering statement:** **Semi-Autoregressive Drafting Specifies Schedulable Prefixes.**

Notebook 17 is the architectural hinge of the `confidence-scheduled-decoding` repository. Notebook 07 treated verification as a scarce engineering resource. Notebook 13 showed how confidence schedules that resource. This notebook asks what kind of draft generation produces prefixes that are worth scheduling.

The central claim is not only that semi-autoregressive drafting can improve draft quality. The stronger specification is that the draft model shapes the confidence profile that the scheduler receives.


## 0. Setup

The notebook writes figures, tables, manifests, and downloadable artifacts into repository-relative folders when available.


In [ ]:
from pathlib import Path
import json
import math
import os
import shutil
import sys
import textwrap
import zipfile
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Robust path handling: run from repo root, notebooks/, or uploaded notebook copy.
CWD = Path.cwd().resolve()
if (CWD / "notebooks").exists() and (CWD / "README.md").exists():
    REPO_ROOT = CWD
elif CWD.name == "notebooks" and (CWD.parent / "README.md").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "17_semi_autoregressive_drafting"
SITE = REPO_ROOT / "site"

for path in [NOTEBOOKS, FIGURES, RESULTS, SITE]:
    path.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(17)

print(f"Repo root: {REPO_ROOT}")
print(f"Figures:   {FIGURES}")
print(f"Results:   {RESULTS}")


## 1. Start here

Notebook 17 connects the scheduler to the drafter.

```text
parallel backbone
        ↓
local sequential head
        ↓
conditional confidence
        ↓
prefix survival
        ↓
verification scheduler
        ↓
accepted tokens
        ↓
higher throughput
```

The engineering question is:

> **How should the draft itself be generated so that scheduling has higher-value prefixes to verify?**

Rather than treating drafting and scheduling as independent modules, DSpark-style semi-autoregressive decoding co-designs them: the draft architecture produces the confidence profile that the verification scheduler exploits.


In [ ]:
roadmap = pd.DataFrame([
    {"notebook": "00", "title": "Context", "question": "Why is decoding expensive?"},
    {"notebook": "07", "title": "Verification Resource", "question": "What resource is scarce?"},
    {"notebook": "13", "title": "Confidence Scheduling", "question": "How should verification be allocated?"},
    {"notebook": "17", "title": "Semi-Autoregressive Drafting", "question": "How should drafts be generated for scheduling?"},
    {"notebook": "23", "title": "Throughput Objective", "question": "What objective should the scheduler optimize?"},
    {"notebook": "29", "title": "Hardware-Aware Scheduling", "question": "How does hardware load change the policy?"},
])
roadmap_path = RESULTS / "notebook_roadmap.csv"
roadmap.to_csv(roadmap_path, index=False)
roadmap


## 2. Why semi-autoregression?

Drafting sits between two constraints.

Autoregressive draft generation preserves local dependencies, but pays a sequential cost for every proposed token.

Parallel draft generation proposes a whole block at once, but later positions cannot condition on the sampled prefix inside the block. This creates suffix decay: the beginning of a draft may be strong, while the end becomes less coherent and less likely to survive target verification.

Semi-autoregressive drafting keeps the heavy computation parallel and adds a lightweight local sequential head. The result is not a full return to autoregression; it is a targeted refinement of the part that matters for scheduling: the conditional prefix.


In [ ]:
# Figure 1: block generation comparison
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.axis("off")
ax.set_title("Why semi-autoregressive drafting?", fontsize=16, pad=16)

rows = [
    ("Autoregressive drafter", "token → token → token → token", "strong local dependence\nsequential draft cost"),
    ("Parallel drafter", "[ token token token token ]", "single draft pass\nweak intra-block conditioning"),
    ("Semi-autoregressive drafter", "parallel block + light local head", "fast draft backbone\nrecover local transitions"),
]

for i, (name, flow, note) in enumerate(rows):
    y = 0.78 - i * 0.28
    ax.text(0.05, y, name, ha="left", va="center", fontsize=12, fontweight="bold")
    ax.text(0.38, y, flow, ha="center", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="black", lw=1.1))
    ax.text(0.72, y, note, ha="left", va="center", fontsize=11)

ax.text(0.5, 0.04,
        "Design goal: keep drafting parallel while restoring enough prefix dependence for scheduling.",
        ha="center", va="center", fontsize=12)

fig.tight_layout()
block_fig = FIGURES / "17_block_generation.png"
fig.savefig(block_fig, dpi=180, bbox_inches="tight")
plt.show()
block_fig


The figure above separates three different design targets. Autoregression maximizes dependency modeling, parallel drafting maximizes draft throughput, and semi-autoregression tries to keep the useful part of both.

For confidence-scheduled decoding, the relevant output is not simply a longer draft. The relevant output is a draft whose early positions form a longer high-survival prefix.


In [ ]:
# Figure 2: architecture flow
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.axis("off")
ax.set_title("Semi-autoregressive architecture flow", fontsize=16, pad=16)

steps = [
    "Target\ncontext",
    "Parallel\nbackbone",
    "Local\nsequential head",
    "Conditional\nconfidence",
    "Schedulable\nprefix",
]
xs = np.linspace(0.08, 0.92, len(steps))
for i, (x, label) in enumerate(zip(xs, steps)):
    ax.text(x, 0.55, label, ha="center", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.55", fc="white", ec="black", lw=1.2))
    if i < len(steps)-1:
        ax.annotate("", xy=(xs[i+1]-0.075, 0.55), xytext=(x+0.075, 0.55),
                    arrowprops=dict(arrowstyle="->", lw=1.8))

ax.text(0.5, 0.18,
        "Heavy computation stays parallel; lightweight local dependence reshapes the confidence profile.",
        ha="center", va="center", fontsize=12)

fig.tight_layout()
arch_fig = FIGURES / "17_architecture_flow.png"
fig.savefig(arch_fig, dpi=180, bbox_inches="tight")
plt.show()
arch_fig


## 3. Confidence profiles become scheduler inputs

Notebook 13 treated confidence as a scheduling variable. Notebook 17 clarifies where that variable comes from.

A drafter does not merely propose token identities. It also creates a confidence profile across the draft positions. That profile determines which prefixes are worth sending to target verification.

In this simplified simulation:

- an autoregressive drafter has stable conditional confidence but pays more draft latency,
- a purely parallel drafter starts strong but decays rapidly,
- a semi-autoregressive drafter preserves much of the parallel advantage while reducing suffix decay.


In [ ]:
# Figure 3: confidence profiles across positions
positions = np.arange(1, 9)
parallel = np.array([0.94, 0.88, 0.80, 0.70, 0.58, 0.47, 0.36, 0.28])
autoreg = np.array([0.86, 0.86, 0.85, 0.85, 0.84, 0.84, 0.83, 0.83])
semi = np.array([0.94, 0.91, 0.88, 0.84, 0.79, 0.72, 0.65, 0.58])

profiles = pd.DataFrame({
    "position": positions,
    "parallel": parallel,
    "autoregressive": autoreg,
    "semi_autoregressive": semi,
})
profiles_path = RESULTS / "confidence_profiles.csv"
profiles.to_csv(profiles_path, index=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(positions, parallel, marker="o", label="parallel")
ax.plot(positions, autoreg, marker="o", label="autoregressive")
ax.plot(positions, semi, marker="o", label="semi-autoregressive")
ax.set_xlabel("Draft position")
ax.set_ylabel("Conditional confidence $c_j$")
ax.set_title("Draft architecture shapes conditional confidence")
ax.set_ylim(0.2, 1.0)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
confidence_fig = FIGURES / "17_confidence_profiles.png"
fig.savefig(confidence_fig, dpi=180, bbox_inches="tight")
plt.show()
confidence_fig


Conditional confidence matters because speculative verification accepts a **prefix**, not arbitrary independent positions. The probability that position \(j\) is useful depends on whether all earlier tokens also survive.

That cumulative event is prefix survival:

\[
a_j=\prod_{k=1}^{j} c_k.
\]

Small differences in conditional confidence therefore compound into larger differences in schedulable prefix length.


In [ ]:
# Figure 4: prefix survival from confidence profiles
survival = pd.DataFrame({"position": positions})
for col in ["parallel", "autoregressive", "semi_autoregressive"]:
    survival[col] = np.cumprod(profiles[col].to_numpy())

survival_path = RESULTS / "prefix_survival.csv"
survival.to_csv(survival_path, index=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(positions, survival["parallel"], marker="o", label="parallel")
ax.plot(positions, survival["autoregressive"], marker="o", label="autoregressive")
ax.plot(positions, survival["semi_autoregressive"], marker="o", label="semi-autoregressive")
ax.axhline(0.35, linestyle="--", linewidth=1.2, label="example scheduling floor")
ax.set_xlabel("Draft position")
ax.set_ylabel("Prefix survival $a_j$")
ax.set_title("Conditional confidence compounds into prefix survival")
ax.set_ylim(0, 1.0)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
survival_fig = FIGURES / "17_prefix_survival.png"
fig.savefig(survival_fig, dpi=180, bbox_inches="tight")
plt.show()
survival_fig


## 4. Draft quality improves throughput

The scheduler can only allocate useful verification if useful prefixes exist.

A semi-autoregressive drafter gives the scheduler more high-survival prefix extensions to choose from. That increases expected accepted length without requiring the system to verify all low-confidence suffix tokens.

The next figure converts confidence profiles into scheduled prefix lengths using a simple survival threshold. This is only an illustrative policy; Notebook 13 already introduced richer budget-aware scheduling. The point here is architectural: better draft profiles create longer schedulable prefixes.


In [ ]:
# Figure 5: prefix gain under an illustrative scheduling floor
thresholds = [0.50, 0.35, 0.20]
rows = []
for threshold in thresholds:
    for method in ["parallel", "autoregressive", "semi_autoregressive"]:
        length = int(np.sum(survival[method].to_numpy() >= threshold))
        rows.append({"survival_floor": threshold, "method": method, "scheduled_prefix_length": length})
prefix_gain = pd.DataFrame(rows)
prefix_gain_path = RESULTS / "scheduled_prefix_gain.csv"
prefix_gain.to_csv(prefix_gain_path, index=False)

fig, ax = plt.subplots(figsize=(8.5, 5))
width = 0.25
x = np.arange(len(thresholds))
methods = ["parallel", "autoregressive", "semi_autoregressive"]
for i, method in enumerate(methods):
    vals = prefix_gain[prefix_gain["method"] == method]["scheduled_prefix_length"].to_numpy()
    ax.bar(x + (i-1)*width, vals, width=width, label=method)

ax.set_xticks(x)
ax.set_xticklabels([f"$a_j \u2265 {t:.2f}$" for t in thresholds])
ax.set_ylabel("Scheduled prefix length")
ax.set_title("Better draft profiles produce longer schedulable prefixes")
ax.legend()
fig.tight_layout()
prefix_gain_fig = FIGURES / "17_prefix_gain.png"
fig.savefig(prefix_gain_fig, dpi=180, bbox_inches="tight")
plt.show()
prefix_gain_fig


A useful engineering proxy combines expected accepted tokens with the drafting and verification costs that produce them:

\[
\Theta(\ell)=
\frac{\sum_{j=1}^{\ell}a_j}
{T_{\rm draft}(\ell)+T_{\rm verify}(\ell)}.
\]

This equation is not meant to replace DSpark's production scheduler. It is the notebook-level bridge to Notebook 23: draft quality, verification length, and throughput must be optimized together.


In [ ]:
# Figure 6: drafting vs throughput proxy
# Illustrative costs: parallel has low draft cost, autoregressive has growing draft cost,
# semi-autoregressive has low base draft cost plus small sequential overhead.
lengths = positions
costs = {
    "parallel": 0.90 + 0.10 * lengths,
    "autoregressive": 0.55 + 0.32 * lengths,
    "semi_autoregressive": 0.95 + 0.14 * lengths,
}
throughput_rows = []
for method in methods:
    a = survival[method].to_numpy()
    for ell in lengths:
        expected_accepts = np.sum(a[:ell])
        total_cost = costs[method][ell-1]
        throughput = expected_accepts / total_cost
        throughput_rows.append({
            "method": method,
            "prefix_length": int(ell),
            "expected_accepts": expected_accepts,
            "total_cost": total_cost,
            "throughput_proxy": throughput,
        })
throughput_df = pd.DataFrame(throughput_rows)
throughput_path = RESULTS / "drafting_vs_throughput.csv"
throughput_df.to_csv(throughput_path, index=False)

fig, ax = plt.subplots(figsize=(8.5, 5))
for method in methods:
    subset = throughput_df[throughput_df["method"] == method]
    ax.plot(subset["prefix_length"], subset["throughput_proxy"], marker="o", label=method)
ax.set_xlabel("Scheduled prefix length $\\ell$")
ax.set_ylabel("Throughput proxy")
ax.set_title("Draft quality and draft cost jointly shape throughput")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
throughput_fig = FIGURES / "17_drafting_vs_throughput.png"
fig.savefig(throughput_fig, dpi=180, bbox_inches="tight")
plt.show()
throughput_fig


## 5. End-to-end confidence-scheduled decoding

The semi-autoregressive drafter should be viewed as part of an end-to-end serving loop.

The draft model produces candidate tokens and confidence structure. The scheduler selects useful prefixes. The target model verifies the scheduled prefix. Accepted tokens then extend the context and begin the next round.


In [ ]:
# Figure 7: generation pipeline
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.axis("off")
ax.set_title("Draft generation creates the scheduler input", fontsize=16, pad=16)

steps = ["Prompt", "Target\ncontext", "Draft\nblock", "Confidence\nprofile", "Scheduled\nprefix", "Target\nverification"]
xs = np.linspace(0.06, 0.94, len(steps))
for i, (x, label) in enumerate(zip(xs, steps)):
    ax.text(x, 0.55, label, ha="center", va="center", fontsize=11.5,
            bbox=dict(boxstyle="round,pad=0.50", fc="white", ec="black", lw=1.1))
    if i < len(steps) - 1:
        ax.annotate("", xy=(xs[i+1]-0.065, 0.55), xytext=(x+0.065, 0.55),
                    arrowprops=dict(arrowstyle="->", lw=1.6))

ax.text(0.5, 0.18,
        "The scheduler does not operate on tokens alone; it operates on confidence-structured draft blocks.",
        ha="center", va="center", fontsize=12)
fig.tight_layout()
pipeline_fig = FIGURES / "17_generation_pipeline.png"
fig.savefig(pipeline_fig, dpi=180, bbox_inches="tight")
plt.show()
pipeline_fig


In [ ]:
# Figure 8: complete decoding stack
fig, ax = plt.subplots(figsize=(7.5, 6.0))
ax.axis("off")
ax.set_title("Confidence-scheduled decoding stack", fontsize=16, pad=16)

stack = [
    "Prompt / context",
    "Draft model",
    "Semi-autoregressive refinement",
    "Confidence head",
    "Verification scheduler",
    "Target model verification",
    "Accepted tokens",
]
y_positions = np.linspace(0.85, 0.15, len(stack))
for i, (y, label) in enumerate(zip(y_positions, stack)):
    ax.text(0.5, y, label, ha="center", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="black", lw=1.1))
    if i < len(stack)-1:
        ax.annotate("", xy=(0.5, y_positions[i+1]+0.045), xytext=(0.5, y-0.045),
                    arrowprops=dict(arrowstyle="->", lw=1.5))

ax.text(0.5, 0.04,
        "Drafting specifies the operating regime available to the scheduler.",
        ha="center", va="center", fontsize=12)
fig.tight_layout()
stack_fig = FIGURES / "17_decoding_stack.png"
fig.savefig(stack_fig, dpi=180, bbox_inches="tight")
plt.show()
stack_fig


## 6. Key equations

Conditional confidence:

\[
c_j=P(y_j\mid y_{<j}).
\]

Prefix survival:

\[
a_j=\prod_{k=1}^{j}c_k.
\]

Expected accepted prefix:

\[
A(\ell)=\sum_{j=1}^{\ell}a_j.
\]

Engineering throughput objective:

\[
\Theta(\ell)=
\frac{A(\ell)}
{T_{\rm draft}(\ell)+T_{\rm verify}(\ell)}.
\]

Notebook 17's contribution is to connect draft architecture to these quantities. The semi-autoregressive drafter reshapes \(c_j\), which reshapes \(a_j\), which changes the scheduler's feasible high-value prefix lengths.


In [ ]:
equations = {
    "conditional_confidence": "c_j = P(y_j | y_<j)",
    "prefix_survival": "a_j = prod_{k=1}^j c_k",
    "expected_accepted_prefix": "A(ell) = sum_{j=1}^ell a_j",
    "throughput_objective": "Theta(ell) = A(ell) / (T_draft(ell) + T_verify(ell))",
}
equations_path = RESULTS / "equations.json"
equations_path.write_text(json.dumps(equations, indent=2), encoding="utf-8")
equations


## 7. Engineering summary

| Layer | Specification | Engineering effect |
|---|---|---|
| Draft backbone | parallel computation | keeps draft latency low |
| Sequential head | local token dependence | improves suffix coherence |
| Confidence profile | per-position scheduling signal | identifies useful prefix extensions |
| Prefix survival | cumulative acceptance estimate | controls verification length |
| Scheduler | budget allocation | routes target compute to high-value prefixes |

The central design shift is that the drafter is not only a proposal generator. It is a producer of schedulable confidence structure.


In [ ]:
summary = pd.DataFrame([
    {"layer": "Draft backbone", "specification": "parallel computation", "engineering_effect": "keeps draft latency low"},
    {"layer": "Sequential head", "specification": "local token dependence", "engineering_effect": "improves suffix coherence"},
    {"layer": "Confidence profile", "specification": "per-position scheduling signal", "engineering_effect": "identifies useful prefix extensions"},
    {"layer": "Prefix survival", "specification": "cumulative acceptance estimate", "engineering_effect": "controls verification length"},
    {"layer": "Scheduler", "specification": "budget allocation", "engineering_effect": "routes target compute to high-value prefixes"},
])
summary_path = RESULTS / "engineering_summary.csv"
summary.to_csv(summary_path, index=False)
summary


## 8. Repository roadmap

```text
00 Context
        ↓
07 Verification Resource
        ↓
13 Confidence Scheduling
        ↓
17 Semi-Autoregressive Drafting
        ↓
23 Throughput Objective
        ↓
29 Hardware-Aware Scheduling
        ↓
37 Operating Regimes
        ↓
43 Resource Allocation
        ↓
49 Adaptive AI Infrastructure
```

Notebook 23 now follows naturally: given confidence profiles produced by the drafter, what throughput objective should the system optimize?


In [ ]:
full_roadmap = pd.DataFrame([
    {"notebook": "00_context", "role": "introduces decoding latency pressure"},
    {"notebook": "07_verification_resource", "role": "frames verification as a scarce resource"},
    {"notebook": "13_confidence_scheduling", "role": "allocates verification using confidence"},
    {"notebook": "17_semi_autoregressive_drafting", "role": "designs drafts that produce schedulable prefixes"},
    {"notebook": "23_throughput_objective", "role": "optimizes accepted tokens against system cost"},
    {"notebook": "29_hardware_aware_scheduling", "role": "adapts scheduling to engine load"},
    {"notebook": "37_operating_regimes", "role": "maps low, medium, and high-load regimes"},
    {"notebook": "43_resource_allocation", "role": "generalizes beyond one decoder"},
    {"notebook": "49_adaptive_ai_infrastructure", "role": "connects scheduling to adaptive serving systems"},
])
full_roadmap_path = RESULTS / "full_repository_roadmap.csv"
full_roadmap.to_csv(full_roadmap_path, index=False)
full_roadmap


## 9. Save notebook manifest

This manifest records the artifacts produced by Notebook 17 v2.


In [ ]:
figure_paths = [
    block_fig,
    arch_fig,
    confidence_fig,
    survival_fig,
    prefix_gain_fig,
    throughput_fig,
    pipeline_fig,
    stack_fig,
]

manifest = {
    "notebook": "17_semi_autoregressive_drafting_v2.ipynb",
    "title": "Semi-Autoregressive Drafting",
    "engineering_statement": "Semi-Autoregressive Drafting Specifies Schedulable Prefixes",
    "outputs": {
        "figures": [str(p.relative_to(REPO_ROOT)) if p.is_relative_to(REPO_ROOT) else str(p) for p in figure_paths],
        "confidence_profiles_csv": str(profiles_path.relative_to(REPO_ROOT)) if profiles_path.is_relative_to(REPO_ROOT) else str(profiles_path),
        "prefix_survival_csv": str(survival_path.relative_to(REPO_ROOT)) if survival_path.is_relative_to(REPO_ROOT) else str(survival_path),
        "scheduled_prefix_gain_csv": str(prefix_gain_path.relative_to(REPO_ROOT)) if prefix_gain_path.is_relative_to(REPO_ROOT) else str(prefix_gain_path),
        "drafting_vs_throughput_csv": str(throughput_path.relative_to(REPO_ROOT)) if throughput_path.is_relative_to(REPO_ROOT) else str(throughput_path),
        "engineering_summary_csv": str(summary_path.relative_to(REPO_ROOT)) if summary_path.is_relative_to(REPO_ROOT) else str(summary_path),
        "equations_json": str(equations_path.relative_to(REPO_ROOT)) if equations_path.is_relative_to(REPO_ROOT) else str(equations_path),
        "roadmap_csv": str(full_roadmap_path.relative_to(REPO_ROOT)) if full_roadmap_path.is_relative_to(REPO_ROOT) else str(full_roadmap_path),
    },
    "next_notebook": "23_throughput_objective.ipynb",
}
manifest_path = RESULTS / "17_semi_autoregressive_drafting_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest


## 10. Download artifacts

Run the final cell to package Notebook 17 outputs for download.


In [ ]:
from IPython.display import FileLink, display

zip_path = RESULTS / "17_semi_autoregressive_drafting_artifacts.zip"
notebook_path = NOTEBOOKS / "17_semi_autoregressive_drafting_v2.ipynb"
fallback_notebook_path = Path.cwd() / "17_semi_autoregressive_drafting_v2.ipynb"

paths_to_include = [
    notebook_path if notebook_path.exists() else fallback_notebook_path,
    RESULTS,
]
paths_to_include.extend(figure_paths)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if path.is_file() and path.resolve() != zip_path.resolve():
                    try:
                        arcname = path.relative_to(REPO_ROOT)
                    except ValueError:
                        arcname = path.name
                    zf.write(path, arcname)
        elif item.exists() and item.resolve() != zip_path.resolve():
            try:
                arcname = item.relative_to(REPO_ROOT)
            except ValueError:
                arcname = item.name
            zf.write(item, arcname)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
